In [1]:
# !rm -rf cache/validation/

In [2]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)

In [3]:
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from coreset_trainer.trainer import CoresetTrainer
import os
import torch

import argparse



In [4]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    linear_coders
)
train_dataset, test_dataset, estimators = load_data_and_estimators()


influence_estimate_path: ./results/influence/LESSEstimator/8192-True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42/tulu-3-sft-olmo-2-mixture-0225-sample_train/tulu-3-sft-olmo-2-mixture-0225-sample_test/estimate.parquet
dirname: ./results/influence/LESSEstimator/8192-True/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42/tulu-3-sft-olmo-2-mixture-0225-sample_train/tulu-3-sft-olmo-2-mixture-0225-sample_test
exists: True
Total test examples: 1000
Unique indices: 1000


In [5]:
from functools import partial
from explanations import KRandom
k_random_types = [partial(KRandom, seed=s) for s in range(9)]
explanation_types = explanation_types + k_random_types

In [6]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)




num_devices = torch.cuda.device_count()







In [7]:
# import dask.dataframe as dd
# from dask.distributed import Client, LocalCluster


# cluster = LocalCluster(
#     n_workers=32,              
#     threads_per_worker=2,     
#     memory_limit='8.75GB',     
# )
# client = Client(cluster)
# client

In [8]:

# device_ids = itertools.cycle(range(num_devices))
# results = []

# from peft import LoraConfig, get_peft_model, PeftModel


# for estimator in estimators:
    

#     explanations = [
#         explanation_type(row.name, estimator)
#         for explanation_type in explanation_types
#         for _, row in estimator.influence_estimate.iloc[test_indices].iterrows()
#     ]
            
        
#     partial_results_dir =  os.path.join("./cache/validation/partial/",
#         estimator.get_config_string(),
#         os.path.basename(estimator.model_path),
#         train_dataset_name,
#         train_dataset_split,
#         test_dataset_name,
#         test_dataset_split,
#         "partial")
#     with ProcessPoolExecutor(max_workers=2*num_devices) as executor:
#         futures = {}
#         for seed in range(1):
#             for ii, explanation in enumerate(explanations):
                
#                 # # random documents to test log_p for in addition to the document beeing explained
#                 # random_test_indices = test_dataset.shuffle(seed=ii).shuffle(seed=seed).select(range(explanation.k))["indices"]
            
#                 text_indices = [explanation.document_idx] #+ random_test_indices
#                 futures[explanation] = executor.submit(
#                         process,
#                         partial_results_dir,
#                         estimator, explanation,
#                         train_dataset.select(explanation.documents), explanation.documents, 
#                         test_dataset.select(text_indices), text_indices,                 
#                         train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, 
#                         next(device_ids),
#                         seed,
#                         ii
#                 )

#         with tqdm(total=len(futures), desc="Explanations", position=0) as pbar:
#             for future in as_completed(futures.values()):
#                 try:
#                     future.result()  
#                 except Exception as e:
#                     logging.error(f"A future failed: {e}\n{traceback.format_exc()}")
#                     raise
#                 finally:
#                     pbar.update(1)

In [9]:
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def copy_folders(folders, local_root="/tmp", max_workers=32):
    def copy_file(src_file, src_root, dst_root):
        rel_path = os.path.relpath(src_file, src_root)
        dst_file = os.path.join(dst_root, rel_path)
        if os.path.exists(dst_file):
            return
        os.makedirs(os.path.dirname(dst_file), exist_ok=True)
        try:
            subprocess.run(["cp", "-p", src_file, dst_file], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Error copying {src_file} to {dst_file}: {e}")

    def chunks(lst, n):
        k, m = divmod(len(lst), n)
        return [lst[i*k + min(i, m):(i+1)*k + min(i+1, m)] for i in range(n)]

    for remote_folder in folders:
        local_folder = os.path.join(local_root, os.path.basename(remote_folder))
        os.makedirs(remote_folder, exist_ok=True)
        os.makedirs(local_folder, exist_ok=True)
        all_files = [os.path.join(root, f) for root, _, files in os.walk(remote_folder) for f in files]
        file_batches = chunks(all_files, max_workers)

        def worker_copy(batch):
            for f in batch:
                copy_file(f, remote_folder, local_folder)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(worker_copy, batch) for batch in file_batches]
            with tqdm(total=len(futures), desc=f"Copying {remote_folder}") as pbar:
                for future in as_completed(futures):
                    future.result()
                    pbar.update(1)


# copy_folders(["cache/scoring", "cache/validation"], local_root="/tmp", max_workers=32)


In [10]:
import glob
import pandas as pd
import os

In [11]:
import pandas as pd
import glob
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def load_or_combine_parquet(partial_folder: str, output_file: str, batch_size: int = 100000) -> pd.DataFrame:
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        df = pd.read_parquet(output_file, engine='pyarrow')
        print(f"Loaded existing combined Parquet: {output_file}")
    else:
        all_files = glob.glob(os.path.join(partial_folder, "**/*.parquet"), recursive=True)
        if len(all_files) == 0:
            raise ValueError(f"No Parquet files found in the folder: {partial_folder}")

        def read_parquet(file):
            df = pd.read_parquet(file, engine='pyarrow')
            df["source_file"] = file  # add column with the file path
            return df

        dfs = []
        print("total", len(all_files))
        for i in range(0, len(all_files), batch_size):
            batch_files = all_files[i:i+batch_size]
            with ThreadPoolExecutor() as executor:
                futures = {executor.submit(read_parquet, f): f for f in batch_files}
                for future in tqdm(as_completed(futures), total=len(futures), desc=f"Reading batch {i//batch_size + 1}"):
                    df_batch = future.result()
                    dfs.append(df_batch)

        df = pd.concat(dfs, ignore_index=True)
        df.to_parquet(output_file, engine='pyarrow', index=False)
        print(f"Combined Parquet saved to: {output_file}")

    return df


# Usage
df_validation = load_or_combine_parquet("cache/validation", "cache/validation_full/full.parquet")


Loaded existing combined Parquet: cache/validation_full/full.parquet


In [12]:
df_scoring = load_or_combine_parquet("cache/scoring/partial", "cache/scoring/full/full.parquet")
import os

df_scoring["linear_coder"] = df_scoring["source_file"].apply(lambda path: os.path.basename(os.path.dirname(os.path.dirname(path))))

Loaded existing combined Parquet: cache/scoring/full/full.parquet


In [13]:
df_validation_random = df_validation[df_validation["explanation_type"].str.contains("random")]
# df_validation_random = df_validation_random.drop(columns=["indices_trained_on"])
df_validation_selections = df_validation[~df_validation["explanation_type"].str.contains("random")]
df_validation_selections = df_validation_selections.drop(columns=["indices_trained_on"])


df_scoring_random = df_scoring[df_scoring["explanation_type"].str.contains("random")]
# df_scoring_random = df_scoring_random.drop(columns=["indices_trained_on"])
df_scoring_selections = df_scoring[~df_scoring["explanation_type"].str.contains("random")]
# df_scoring_selections = df_scoring_selections.drop(columns=["indices_trained_on"])

In [14]:
df_scoring_random[df_scoring_random["explanation_type"] == "1 random examples with seed 42"]["mse"].mean()

0.00014256990647172697

In [15]:
df_validation_selections.groupby("explanation_type").count()

,model,estimator,document_idx,train_dataset,train_split,test_dataset,test_split,indices_target_document,delta_target_document,source_file
explanation_type,,,,,,,,,,
The test instance (as a sanity check),1027,1027,1027,1027,1027,1027,1027,1027,1027,1027


In [28]:
df_validation_random.groupby("explanation_type").mean(numeric_only=True)

,document_idx,indices_target_document,delta_target_document
explanation_type,,,
1 random examples with seed 10,499.5,499.5,-11.610860
1 random examples with seed 3,499.5,499.5,-4.422440
1 random examples with seed 42,499.5,499.5,-17.252033
1 random examples with seed 6,33.0,33.0,-15.830036
1 random examples with seed 9,499.5,499.5,-16.415355
2 random examples with seed 3,499.5,499.5,-4.323056
2 random examples with seed 9,499.5,499.5,-16.492694
25 random examples with seed 42,21.5,21.5,-62.381037
25 random examples with seed 6,127.0,127.0,-3.718789


In [30]:
df_validation_selections.groupby("explanation_type").mean(numeric_only=True)

,document_idx,indices_target_document,delta_target_document
explanation_type,,,
The test instance (as a sanity check),486.709834,486.709834,-15.457376


In [17]:
import re
import pandas as pd

pattern = r"Top-(\d+)"
group_cols = [
    "model",
    "estimator",
    "train_dataset",
    "train_split",
    "test_dataset",
    "test_split",
    # "LESSEstimator: normalize",
    "document_idx"
]

results = []
for linear_coder in linear_coders:
    linear_coder = linear_coder.__name__
    for explanation_type in df_validation_selections["explanation_type"].unique():
        k = int(re.match(pattern, explanation_type).group(1)) if "The test instance (as a sanity check)" not in explanation_type else 1

        # Only random runs matching this k
        random_run_names = df_validation_random[
            df_validation_random["explanation_type"].str.contains(f"{k} random")
        ]["explanation_type"].unique()

        for random_run_name in random_run_names:

            # Filter and drop duplicate rows per document
            sel = df_validation_selections[(df_validation_selections["explanation_type"] == explanation_type)]
            rnd = df_validation_random[(df_validation_random["explanation_type"] == random_run_name)]
            scores_sel = df_scoring_selections[(df_scoring_selections["explanation_type"] == explanation_type) & (df_scoring_selections["linear_coder"] == linear_coder)]
            scores_rnd = df_scoring_random[(df_scoring_random["explanation_type"] == random_run_name) &(df_scoring_random["linear_coder"] == linear_coder)]

            # Merge selection and random results on group_cols
            r = pd.merge(
                sel,
                rnd,
                on=group_cols,
                suffixes=("_selection", "_random"),
                how="inner"
            )

            # Merge pred_gain columns
            r = pd.merge(
                r,
                scores_sel[group_cols + ["pred_gain", "mse"]].rename(columns={"pred_gain": "pred_gain_selection","mse": "mse_selection"}),
                on=group_cols,
                how="left"
            )
            r = pd.merge(
                r,
                scores_rnd[group_cols + ["pred_gain","mse"]].rename(columns={"pred_gain": "pred_gain_random", "mse": "mse_random"}),
                on=group_cols,
                how="left"
            )



            if len(r) == 0:
                continue
            # display(r[[col for col in r.columns if col != 'indices_trained_on']].nunique())
            # display(r[[col for col in r.columns if col != 'indices_trained_on']].count())
            # display(len(scores_sel))
            # display(scores_sel[group_cols + ["pred_gain", "mse"]].rename(columns={"pred_gain": "pred_gain_selection","mse": "mse_selection"}).count())
            # raise Error

            # Compute summary statistics per random run
            results.append({
                "linear_coder": linear_coder,
                "explanation_type": explanation_type,
                "random_run": random_run_name,
                "k": k,
                "validation_score": (r["delta_target_document_selection"] >= r["delta_target_document_random"]).mean(skipna=True),
                "pred_gain_selection": r["pred_gain_selection"].mean(skipna=True),
                "pred_gain_random": r["pred_gain_random"].mean(skipna=True),
                "mse_selection": r["mse_selection"].mean(skipna=True),
                "mse_random": r["mse_random"].mean(skipna=True),
                "count": len(r),
                **r.iloc[0][[col for col in group_cols if col != "document_idx"]].to_dict(),
            })
            


results_df = pd.DataFrame(results)
results_df


,linear_coder,explanation_type,random_run,k,validation_score,pred_gain_selection,pred_gain_random,mse_selection,mse_random,count,model,estimator,train_dataset,train_split,test_dataset,test_split
0,MSECoderProjUSimpSparse,The test instance (as a sanity check),1 random examples with seed 42,1,0.546000,1.061091,0.965342,0.000156,0.000168,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
1,MSECoderProjUSimpSparse,The test instance (as a sanity check),1 random examples with seed 3,1,0.195000,1.061091,0.933072,0.000156,0.000170,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
2,MSECoderProjUSimpSparse,The test instance (as a sanity check),1 random examples with seed 9,1,0.556000,1.061091,0.764879,0.000156,0.000199,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
3,MSECoderProjUSimpSparse,The test instance (as a sanity check),1 random examples with seed 10,1,0.406000,1.061091,1.025714,0.000156,0.000164,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
4,MSECoderProjUSimpSparse,The test instance (as a sanity check),1 random examples with seed 6,1,0.537313,1.020536,1.257651,0.000151,0.000125,67,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
5,MSECoderProjUSimp,The test instance (as a sanity check),1 random examples with seed 42,1,0.546000,1.061091,0.965342,0.000156,0.000168,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
6,MSECoderProjUSimp,The test instance (as a sanity check),1 random examples with seed 3,1,0.195000,1.061091,0.933072,0.000156,0.000170,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
7,MSECoderProjUSimp,The test instance (as a sanity check),1 random examples with seed 9,1,0.556000,1.061091,0.764879,0.000156,0.000199,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
8,MSECoderProjUSimp,The test instance (as a sanity check),1 random examples with seed 10,1,0.406000,1.061091,1.025714,0.000156,0.000164,1000,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test
9,MSECoderProjUSimp,The test instance (as a sanity check),1 random examples with seed 6,1,0.537313,1.020536,1.257651,0.000151,0.000125,67,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,LESSEstimator: normalize=True,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test


In [18]:
results_df.groupby(["linear_coder", "explanation_type"]).mean(numeric_only=True)

,,k,validation_score,pred_gain_selection,pred_gain_random,mse_selection,mse_random,count
linear_coder,explanation_type,,,,,,,
KLTCoder,The test instance (as a sanity check),1.0,0.448063,0.996884,0.996884,0.000147,0.000147,813.4
MSECoder,The test instance (as a sanity check),1.0,0.448063,1.427298,1.512443,0.000107,0.000101,813.4
MSECoderNNLSL2,The test instance (as a sanity check),1.0,0.448063,1.427248,1.512438,0.000107,0.000101,813.4
MSECoderProjUSimp,The test instance (as a sanity check),1.0,0.448063,1.052980,0.989331,0.000155,0.000165,813.4
MSECoderProjUSimpSparse,The test instance (as a sanity check),1.0,0.448063,1.052980,0.989331,0.000155,0.000165,813.4
MSECoderProjUSimpSparseSoftThresh,The test instance (as a sanity check),1.0,0.448063,1.052980,0.989331,0.000155,0.000165,813.4


In [26]:
from scipy.stats import ttest_rel

results = []

for (k, exp_type,linear_coder), group in results_df.groupby(['k', 'explanation_type', 'linear_coder']):
    # paired test: selection vs random
    t_stat, p_value = ttest_rel(group['validation_score'], group['pred_gain_random'], alternative='greater')
    results.append({'k': k,"linear_coder":linear_coder, 'explanation_type': exp_type, 't_stat': t_stat, 'p_value': p_value})

pd.DataFrame(results)



,k,linear_coder,explanation_type,t_stat,p_value
0,1,KLTCoder,The test instance (as a sanity check),-7.933888,0.999317
1,1,MSECoder,The test instance (as a sanity check),-9.009700,0.999580
2,1,MSECoderNNLSL2,The test instance (as a sanity check),-9.009668,0.999580
3,1,MSECoderProjUSimp,The test instance (as a sanity check),-5.381830,0.997119
4,1,MSECoderProjUSimpSparse,The test instance (as a sanity check),-5.381830,0.997119
5,1,MSECoderProjUSimpSparseSoftThresh,The test instance (as a sanity check),-5.381830,0.997119


In [20]:
from scipy.stats import wilcoxon
w_stat, p_value = wilcoxon(group['validation_score'], group['pred_gain_random'], alternative='greater')
w_stat

0.0

In [21]:
p_value

0.999999135525782

In [22]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Drop NaNs
df = results_df.dropna(subset=["validation_score", "pred_gain_selection", "pred_gain_random"])

# Mixed-effects model
# 'random_run' is a grouping variable (random intercept)
# You can optionally include 'explanation_type' as another random effect
md = smf.mixedlm(
    "validation_score ~ pred_gain_selection + pred_gain_random",
    df,
    groups=df["random_run"]
)
mdf = md.fit()
print(mdf.summary())


            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: validation_score
No. Observations: 30      Method:             REML            
No. Groups:       5       Scale:              0.0000          
Min. group size:  6       Log-Likelihood:     302.0370        
Max. group size:  6       Converged:          Yes             
Mean group size:  6.0                                         
--------------------------------------------------------------
                    Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept            0.448    0.027 16.889 0.000  0.396  0.500
pred_gain_selection  0.000    0.000  0.000 1.000 -0.000  0.000
pred_gain_random    -0.000    0.000 -0.000 1.000 -0.000  0.000
Group Var            0.004 2060.007                           



/root/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [23]:
corr_selection = results_df['validation_score'].corr(results_df['pred_gain_selection'])
corr_random = results_df['validation_score'].corr(results_df['pred_gain_random'])

print(f"Correlation with pred_gain_selection: {corr_selection:.3f}")
print(f"Correlation with pred_gain_random: {corr_random:.3f}")


Correlation with pred_gain_selection: -0.031
Correlation with pred_gain_random: -0.050


In [24]:
import statsmodels.api as sm

# Validation score ~ pred_gain_selection
X1 = sm.add_constant(results_df['pred_gain_selection'])
model1 = sm.OLS(results_df['validation_score'], X1).fit()
print(model1.summary())

# Validation score ~ pred_gain_random
X2 = sm.add_constant(results_df['pred_gain_random'])
model2 = sm.OLS(results_df['validation_score'], X2).fit()
print(model2.summary())


                            OLS Regression Results                            
Dep. Variable:       validation_score   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.035
Method:                 Least Squares   F-statistic:                   0.02676
Date:                Fri, 26 Sep 2025   Prob (F-statistic):              0.871
Time:                        17:46:37   Log-Likelihood:                 16.893
No. Observations:                  30   AIC:                            -29.79
Df Residuals:                      28   BIC:                            -26.98
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.4749    

In [25]:
import statsmodels.api as sm
import pandas as pd

results = []

for keys, group in df.groupby(["explanation_type", "model", "estimator", "linear_coder"]):
    x = -pd.to_numeric(group["mse"], errors="coerce")  # negate MSE so lower is better
    y = pd.to_numeric(group["delta_target_document"], errors="coerce")
    
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    

    X = sm.add_constant(pd.DataFrame({"neg_mse": x_clean}))
    model = sm.OLS(y_clean, X).fit()
    
    results.append({
        "explanation_type": keys[0],
        "model": keys[1],
        "estimator": keys[2],
        "linear_coder": keys[3],
        "coef": model.params.get("neg_mse", None),       
        "p_value": model.pvalues.get("neg_mse", None),  
        "r_squared": model.rsquared
    })

results_df = pd.DataFrame(results)
results_df


KeyError: 'mse'

In [ ]:
import statsmodels.api as sm

results = []


for keys, group in df.groupby(["explanation_type", "model", "estimator", "linear_coder"]):
    x = pd.to_numeric(group["pred_gain"], errors="coerce")
    y = pd.to_numeric(group["delta_target_document"], errors="coerce")
    
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    

    X = sm.add_constant(x_clean)
    model = sm.OLS(y_clean, X).fit()
    
    results.append({
        "explanation_type": keys[0],
        "model": keys[1],
        "estimator": keys[2],
        "linear_coder": keys[3],
        "coef": model.params.get("pred_gain", None),
        "p_value": model.pvalues.get("pred_gain", None),
        "r_squared": model.rsquared
    })


results_df = pd.DataFrame(results)
results_df
